# gradient-routing-slm: one run on a Kaggle T4
Settings: **Accelerator = GPU T4** (x1 or x2, only one is used), **Internet = On**. Edit the first cell, then Run All.
The last cell writes a zip to `/kaggle/working/`. Unzip it into `results/<name>/` in the repo, using the zip's name.

In [ ]:
ROUTING_MODE = "dense"   # dense (baseline), batch (v0), token (v1), label (oracle), assembly (v2)
SEED = 0                 # model init and batch order; the dataset and val sets stay fixed
MAX_ITERS = 5000

# assembly mode only; other values give the run its own name, e.g. assembly_r0.5_s0
RESOLUTION = 1.0         # Leiden resolution: higher means more, smaller assemblies
FLOOR = 0.1              # lowest plasticity any neuron can get
PLASTICITY = "rank"      # rank or sigmoid

LR_SCALE = 1.0           # dense control: set to the value in an assembly run's summary.txt

POSTHOC = True           # after training, also cluster the finished model post hoc and ablate (the dense control)
POSTHOC_CKPT = ""        # set to an uploaded checkpoint, e.g. "/kaggle/input/dense-s0/ckpt.pt", to ONLY run the post-hoc
                         # evaluation on it and skip training
REPO_URL = "https://github.com/sandwichhlord/gradient-routing-slm.git"

In [ ]:
!pip install -q -U tiktoken datasets igraph leidenalg

In [ ]:
!rm -rf /tmp/repo && git clone --depth 1 {REPO_URL} /tmp/repo
%cd /tmp/repo
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import json, os, shutil, subprocess
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 in the notebook settings"


def stream(cmd, log, env=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    return p.wait()


tag = ""
if RESOLUTION != 1.0: tag += f"_r{RESOLUTION}"
if FLOOR != 0.1: tag += f"_f{FLOOR}"
if PLASTICITY != "rank": tag += f"_{PLASTICITY}"
if LR_SCALE != 1.0: tag += f"_lr{LR_SCALE}"
RUN = f"posthoc_{os.path.basename(os.path.dirname(POSTHOC_CKPT))}" if POSTHOC_CKPT else f"{ROUTING_MODE}{tag}_s{SEED}"
OUT = f"out_{RUN}"
os.makedirs(OUT, exist_ok=True)
with open("run.log", "w") as log:
    if POSTHOC_CKPT:
        rc = stream(["python", "data/prepare.py"], log)
        if rc == 0:
            rc = stream(["python", "eval/ablate_assemblies.py", "--ckpt", POSTHOC_CKPT, "--out_dir", OUT, "--save_dir", OUT,
                         "--source", "posthoc", "--resolution", str(RESOLUTION)], log)
        if rc == 0:
            rc = stream(["python", "eval/assembly_probe.py", "--ckpt", POSTHOC_CKPT, "--out_dir", OUT,
                         "--assignments", f"{OUT}/assemblies_posthoc.json"], log)
    else:
        env = {**os.environ, "ROUTING_MODE": ROUTING_MODE, "SEED": str(SEED), "MAX_ITERS": str(MAX_ITERS), "OUT_DIR": OUT,
               "RESOLUTION": str(RESOLUTION), "FLOOR": str(FLOOR), "PLASTICITY": PLASTICITY, "LR_SCALE": str(LR_SCALE),
               "POSTHOC": "1" if POSTHOC else "0"}
        rc = stream(["bash", "run_all.sh"], log, env)
shutil.copy("run.log", f"{OUT}/run.log")
if os.path.exists("data/out/meta.json"):
    shutil.copy("data/out/meta.json", f"{OUT}/data_meta.json")
    mode = json.load(open("data/out/meta.json"))["mode"]
    print()
    print("DATA MODE:", mode, "" if mode == "real" else " <-- WARNING: HF download failed, results are on synthetic data!")
print("exit code:", rc, "(the last cell still zips whatever was produced)")

In [ ]:
from IPython.display import Image, display
for name in ["loss_curves.png", "ablation_heatmap.png", "routing_by_domain.png", "assembly_health.png", "assembly_sizes.png",
             "assembly_heatmap.png", "posthoc/assembly_sizes.png", "posthoc/assembly_heatmap.png"]:
    if os.path.exists(f"{OUT}/{name}"):
        print(name)
        display(Image(f"{OUT}/{name}"))
for name in ["summary.txt", "ablation_summary.txt", "assembly_summary.txt", "posthoc/assembly_summary.txt", "samples.txt"]:
    if os.path.exists(f"{OUT}/{name}"):
        print("---", name)
        print(open(f"{OUT}/{name}").read())

In [ ]:
zip_path = shutil.make_archive(f"/kaggle/working/{RUN}", "zip", OUT)
print("wrote", zip_path, f"({os.path.getsize(zip_path) / 1e6:.0f} MB)")
assert rc == 0, f"the run failed with exit code {rc}: see run.log inside the zip"